In [1]:
"""
High/Medium/Low CI Theme Miner for Android Instrumentation (LOG FETCH + STRATUM GATING)
--------------------------------------------------------------------------------------
What this script does
- Reads your stratified sample (html_url + stratum) and strict weights from cohort_table.csv.
- Fetches GitHub Actions logs (token rotation, backoff) within a time window aligned to your snapshot.
- Scans logs for High/Medium/Low themes, but:
  * High themes always counted.
  * Medium/Low are only counted if the repo has instrumentation context based on STRATUM (Y/B/A).
    - De-noising for Med/Low: require >=2 matching lines OR hits in >=2 runs.
- Collects YAML hints (remote workflows + local configs) for audit context only (not gating).
- Outputs weighted counts, by-provider breakdown, examples, and an audit table.

Your key adjustments included:
1) Trust STRATUM (Y/B/A) for instrumentation context gating instead of YAML hints.
2) Expand log search period to 90 days BEFORE clone date (no drift after clone date).
3) Keep progress logs, early-stop, and token rotation.

Outputs (saved in OUT_DIR):
  - challenge_counts_weighted_high_only.csv
  - challenge_counts_by_provider_weighted_high_only.csv
  - challenge_examples_high_only.csv
  - challenge_run_audit_high_only.csv
"""

import os, re, io, zipfile, json, time, random
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, time as dtime, timedelta, timezone

# ======================= CONFIG =======================
CONFIG_DIR        = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
SAMPLED_CSV_DIR   = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified_SampleV2.0")
SAMPLED_CSV_NAME  = "stratified_sample_moe10.csv"   # <<— use this exact file
COHORT_FILE_NAME  = "cohort_table.csv"
OUT_DIR           = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\CI_Taxomony_Analysis")


# Tokens env file (both casings tried)
TOKENS_ENV_PATH   = Path(r"C:\GitHub\Android-Mobile-Apps\all_tokens.env")
if not TOKENS_ENV_PATH.exists():
    TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

# Remote fetch controls
ENABLE_GITHUB_FETCH = True
MAX_RUNS_PER_REPO   = 60       # ↑ coverage
REQUEST_TIMEOUT     = 30
REQUESTS_MAX_RETRIES= 3
BACKOFF_BASE_SEC    = 2.0

# Early stop: stop scanning more runs for a repo once a hit is confirmed
EARLY_STOP_ON_HIT = True
EARLY_STOP_MODE   = "any"      # "any" (high/med/low) or "high-only"

# Time window around clone date (align logs to snapshot)
USE_CUTOFF          = True
CLONE_DATE_LOCAL    = "2025-08-10"   # your clone date
WINDOW_DAYS_BEFORE  = 90             # include logs up to 90 days BEFORE clone date
WINDOW_DAYS_AFTER   = 0              # keep 0 to avoid drift after snapshot
LOCAL_TZ            = "America/Toronto"

# Progress logging
VERBOSE = True
PROGRESS_EVERY = 1
SHOW_HIT_SNIPPET_COUNT = True

# ======================= THEMES =======================
THEMES = {
  # High (hard runtime failures, always counted)
  "EMULATOR_BOOT_TIMEOUT": ["device offline", "boot completed timeout", "emulator: ERROR", "emulator: Panic", "Adb connection refused"],
  "HW_ACCEL/KVM_MISSING": ["/dev/kvm permission denied", "KVM is required", "accel not installed", "HAXM is not installed", "Hypervisor.framework is required"],
  "MISSING_EMULATOR_BIN": ["emulator: not found", "No emulator installed"],
  "ADB_ISSUE": ["ADB server didn't ACK", "ADB server version mismatch", "error: device offline", "more than one device/emulator"],
  "X11/HEADLESS": ["xvfb-run: error", "Cannot open display", "DISPLAY not set"],
  "VENDOR_LAB_QUOTA": ["Quota exceeded", "Billing account not configured", "insufficient tokens for"],

  # Medium (config/setup + infra)
  "SDK_LICENSES": ["You have not accepted the license agreements", "licenses not accepted", "SDK licenses not accepted"],
  "IMAGE_NOT_FOUND": [r"Package .* was not found", r"failed to find target with hash string", r"system-images;android-\d+.*not found"],
  "JAVA/GRADLE/AGP_VERSION_DRIFT": ["Minimum supported Gradle is", "This version of the Android Gradle plugin requires", "Unsupported Java", "Kotlin version .* is not compatible"],
  "ANDROIDX_TEST_MISMATCH": ["Could not resolve androidx.test", "Duplicate class .* found in modules", "conflict with dependency 'androidx.test'"],
  "NETWORK_FLAKE": ["Connection reset by peer", "TLS handshake timeout", "temporary failure in name resolution", "network is unreachable"],
  "SECRETS_PERMS": ["Permission denied", "No such file or directory .*keystore", "secrets.* not set", "Missing .* environment variable"],
  "GENERIC_TIMEOUT/CANCEL": ["job timed out", "The operation was canceled", "timeout exceeded"],

  # Low (weak signals)
  "TEST_FLAKY_SIGNAL": [r"Flaky", r"retrying test", r"java\.lang\.AssertionError", r"org\.junit\.ComparisonFailure"],
}

HIGH_THEMES   = {"EMULATOR_BOOT_TIMEOUT","HW_ACCEL/KVM_MISSING","MISSING_EMULATOR_BIN","ADB_ISSUE","X11/HEADLESS","VENDOR_LAB_QUOTA"}
MEDIUM_THEMES = {"SDK_LICENSES","IMAGE_NOT_FOUND","JAVA/GRADLE/AGP_VERSION_DRIFT","ANDROIDX_TEST_MISMATCH","NETWORK_FLAKE","SECRETS_PERMS","GENERIC_TIMEOUT/CANCEL"}
LOW_THEMES    = {"TEST_FLAKY_SIGNAL"}

# Medium/Low de-noising + gating
INCLUDE_MEDIUM = True
INCLUDE_LOW    = True
MEDLOW_MIN_LINE_HITS     = 2   # >=2 matching lines ...
MEDLOW_MIN_RUNS_WITH_HIT = 2   # ... OR hits in >=2 runs
# GATING SOURCE: STRATUM ONLY (Y/B/A)
# YAML hints are collected for context but NOT used to gate Medium/Low.

# Simple YAML hints (for audit context only)
YAML_HINTS = [
    "reactivecircus/android-emulator-runner",
    "sdkmanager",
    "avdmanager",
    "emulator -avd",
    "gcloud firebase test android run",
    "adb", "androidTest", "connectedAndroidTest"
]
TEXT_EXTS = {".yml",".yaml",".log",".txt",".json",".xml",".gradle",".kts",".properties",".sh",".bat",".cfg",".conf",".ini",".md"}

# ======================= HELPERS =======================
def ts(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
def banner(msg):   print(f"[{ts()}] ===== {msg} =====", flush=True) if VERBOSE else None
def note(msg):     print(f"[{ts()}] {msg}", flush=True)
def warn(msg):     print(f"[{ts()}] [WARN] {msg}", flush=True)
def info(msg):     print(f"[{ts()}] [INFO] {msg}", flush=True)

def load_tokens_from_env_file(path: Path) -> list:
    tokens = {}
    if path and path.exists():
        for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = raw.strip()
            if not line or line.startswith(("#",";")) or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k.upper().startswith("GITHUB_TOKEN_"):
                try: idx = int(k.split("_")[-1])
                except Exception: idx = 999
                tokens[idx] = v
    return [tokens[i] for i in sorted(tokens.keys()) if tokens[i]]

class TokenRotator:
    def __init__(self, tokens: list[str]): self.tokens, self.i = (tokens or []), 0
    def current(self): return (self.tokens[self.i] if self.tokens else None)
    def rotate(self): 
        if self.tokens: self.i = (self.i + 1) % len(self.tokens)
    def headers(self, allow_auth=True):
        base = {"Accept":"application/vnd.github+json","User-Agent":"challenge-miner/high-med-low/1.0"}
        tok = self.current()
        if allow_auth and tok: base["Authorization"] = f"Bearer {tok}"
        return base

def gh_get(url, rot: TokenRotator, allow_auth=True, timeout=REQUEST_TIMEOUT):
    import requests
    tries = 0
    while True:
        tries += 1
        resp = requests.get(url, headers=rot.headers(allow_auth=allow_auth), timeout=timeout)
        diag = {"status": resp.status_code, "rate_remaining": resp.headers.get("X-RateLimit-Remaining"),
                "rate_reset": resp.headers.get("X-RateLimit-Reset"), "message": None}
        try:
            j = resp.json()
            if isinstance(j, dict) and "message" in j: diag["message"] = j["message"]
        except Exception: pass
        resp._diag = diag

        if resp.status_code in (200,201,204): return resp
        if resp.status_code in (403,429):
            warn(f"Rate/403 on {url}. Rotating token. Diag={diag}")
            rot.rotate()
            if tries >= REQUESTS_MAX_RETRIES * max(1, len(rot.tokens)): return resp
            sleep_s = BACKOFF_BASE_SEC * (2 ** (tries-1)) + random.random()
            time.sleep(min(sleep_s, 30)); continue
        if resp.status_code == 401 and allow_auth:
            return gh_get(url, rot, allow_auth=False, timeout=timeout)
        return resp

def owner_repo_from_url(url: str):
    try:
        u = (url or "").strip().strip("/")
        if "github.com/" in u:
            tail = u.split("github.com/", 1)[1].strip("/")
            parts = tail.split("/")
            if len(parts) >= 2: return parts[0].lower(), parts[1].lower()
    except Exception: pass
    return None, None

def full_name_from_html_url(url: str) -> str | None:
    o, r = owner_repo_from_url(url); return f"{o}/{r}" if (o and r) else None

def full_name_from_saved_filename(path: Path) -> str | None:
    base = path.name
    m = re.match(r"^([A-Za-z0-9_.-]+)\.([A-Za-z0-9_.-]+)__", base)
    if m:
        owner, repo = m.group(1), m.group(2)
        return f"{owner.replace(' ','').lower()}/{repo.replace(' ','').lower()}"
    return None

# ---- time window helpers ----
try:
    from zoneinfo import ZoneInfo
except Exception:
    ZoneInfo = None

def parse_gh_ts(ts: str):
    if not ts: return None
    try: return datetime.strptime(ts, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    except Exception: return None

def window_utc_from_local(center_date_str: str, days_before: int, days_after: int, tz_name: str):
    try:
        center = datetime.strptime(center_date_str, "%Y-%m-%d").date()
    except Exception:
        return (None, None, True)
    start_date = center - timedelta(days=days_before)
    end_date   = center + timedelta(days=days_after)
    if ZoneInfo is None:
        # fallback: compare by DATE only
        return (start_date, end_date, True)
    tz = ZoneInfo(tz_name)
    start_local = datetime.combine(start_date, dtime(0,0,0)).replace(tzinfo=tz)
    end_local   = datetime.combine(end_date,   dtime(23,59,59)).replace(tzinfo=tz)
    return (start_local.astimezone(timezone.utc), end_local.astimezone(timezone.utc), False)

# ======================= REMOTE FETCH =======================
def list_workflow_runs(rot: TokenRotator, owner: str, repo: str, page: int, per_page: int = 100):
    return gh_get(f"https://api.github.com/repos/{owner}/{repo}/actions/runs?per_page={per_page}&page={page}",
                  rot, allow_auth=True)

def fetch_log_text_for_run(rot: TokenRotator, logs_url: str):
    resp = gh_get(logs_url, rot, allow_auth=True)
    if resp.status_code != 200: return None, resp._diag
    try:
        zf = zipfile.ZipFile(io.BytesIO(resp.content))
    except zipfile.BadZipFile:
        return None, resp._diag
    texts = []
    for name in zf.namelist():
        try:
            with zf.open(name) as f:
                texts.append(f.read().decode("utf-8", errors="ignore"))
        except Exception:
            continue
    return "\n".join(texts), None

def fetch_workflow_yamls(rot: TokenRotator, owner: str, repo: str):
    """Context only (not gating)."""
    yamls, err = [], None
    r = gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/.github/workflows", rot, allow_auth=True)
    if r.status_code == 200:
        for item in r.json():
            if item.get("type") == "file" and item.get("name", "").lower().endswith((".yml",".yaml")):
                raw_url = item.get("download_url")
                if not raw_url: continue
                fr = gh_get(raw_url, rot, allow_auth=True)
                if fr.status_code == 200: yamls.append(fr.text)
    else:
        err = r._diag
    return "\n".join(yamls), err

# ======================= SCANNING =======================
PAT_MAP = {k: [re.compile(p, re.I) for p in v] for k, v in THEMES.items()}

def scan_text_for_themes(text: str):
    """
    Returns:
      - found: list of themes that matched at least one line
      - hits:  list of (theme, line) for each matching line (truncated to 500 chars)
    """
    found, hits = set(), []
    if not text: return [], []
    for theme, regs in PAT_MAP.items():
        for line in text.splitlines():
            for rx in regs:
                if rx.search(line):
                    found.add(theme)
                    hits.append((theme, line[:500]))
                    break
            if theme in found:
                # continue scanning other themes; we still collect per-line hits for Med/Low de-noising
                continue
    return list(found), hits

def scan_yaml_hints(text: str):
    L = (text or "").lower()
    return [h for h in YAML_HINTS if h.lower() in L]

# ---------- STRATUM-BASED GATING ----------
STRATUM_RX = re.compile(r"^Y([01])_B([01])_A([01])$")

def parse_stratum_bits(s: str) -> tuple[int,int,int]:
    if not isinstance(s, str):
        return (0,0,0)
    m = STRATUM_RX.match(s.strip())
    if not m:
        return (0,0,0)
    return (int(m.group(1)), int(m.group(2)), int(m.group(3)))

def has_instru_context_from_stratum(stratum: str, at_pred: int | None = None) -> bool:
    y, b, a = parse_stratum_bits(stratum)
    if (y == 1) or (b == 1) or (a == 1):
        return True
    return bool(int(at_pred or 0) == 1)

# ========== Med/Low de-noising rule ==========
def _theme_counts_as_hit(theme: str, theme_line_hits: dict, theme_run_ids: dict, high: bool=False) -> bool:
    """
    High themes: any single hit counts.
    Medium/Low: require >= MEDLOW_MIN_LINE_HITS (line-level matches) OR
                >= MEDLOW_MIN_RUNS_WITH_HIT (distinct runs with a match).
    """
    if theme not in theme_line_hits and theme not in theme_run_ids:
        return False
    if (high or theme in HIGH_THEMES):
        return (theme_line_hits.get(theme, 0) >= 1)
    # Med/Low rule:
    line_hits = theme_line_hits.get(theme, 0)
    run_hits  = len(theme_run_ids.get(theme, set()))
    return (line_hits >= MEDLOW_MIN_LINE_HITS) or (run_hits >= MEDLOW_MIN_RUNS_WITH_HIT)

# ======================= DATA LOADING =======================
def auto_pick_stratified_csv(folder: Path) -> Path:
    cands = sorted([p for p in folder.glob("*.csv") if "stratified" in p.name.lower()])
    if not cands:
        cands = sorted(folder.glob("*.csv"))
    if not cands:
        raise FileNotFoundError(f"No CSV found in {folder}")
    return cands[0]

def load_sample_and_weights(folder: Path, sample_name: str | None, cohort_name: str) -> pd.DataFrame:
    # Sample
    samp_path = (folder / sample_name) if sample_name else auto_pick_stratified_csv(folder)
    sample = pd.read_csv(samp_path)

    if "html_url" not in sample.columns:
        for alt in ["url","repo_url","html_urls","html"]:
            if alt in sample.columns:
                sample = sample.rename(columns={alt:"html_url"}); break
    if "stratum" not in sample.columns:
        for alt in ["Stratum","strata","group"]:
            if alt in sample.columns:
                sample = sample.rename(columns={alt:"stratum"}); break
    if "html_url" not in sample.columns or "stratum" not in sample.columns:
        raise ValueError("Sample CSV must include 'html_url' and 'stratum' columns.")

    sample["full_name"] = sample["html_url"].apply(full_name_from_html_url)

    # Keep AT_pred if present; else 0
    if "AT_pred" not in sample.columns:
        sample["AT_pred"] = 0

    # Cohort (strict)
    cohort_path = folder / cohort_name
    if cohort_name.lower() != "cohort_table.csv":
        raise ValueError("COHORT_FILE_NAME must be 'cohort_table.csv'.")
    if not cohort_path.exists():
        raise FileNotFoundError(f"Required cohort table not found: {cohort_path}")

    cohort = pd.read_csv(cohort_path)
    if "stratum" not in cohort.columns:
        for alt in ["Stratum","strata","group"]:
            if alt in cohort.columns:
                cohort = cohort.rename(columns={alt:"stratum"}); break
    if "stratum" not in cohort.columns:
        raise ValueError("cohort_table.csv must include a 'stratum' column.")

    # Determine weights: prefer 'weight'; else N/n_sample; else 1.0
    weight_col = None
    for c in cohort.columns:
        if c.strip().lower() in {"weight","w","wt"}:
            weight_col = c; break
    if weight_col is None:
        lc_map = {c.lower(): c for c in cohort.columns}
        pop_col  = next((lc_map[k] for k in ["n","n_total","n_population","population","pop","n_pop","total","size"] if k in lc_map), None)
        samp_col = next((lc_map[k] for k in ["n_sample","sample","sample_n","n_samp","sample_size"] if k in lc_map), None)
        if pop_col is not None and samp_col is not None:
            cohort["__weight__"] = (
                pd.to_numeric(cohort[pop_col], errors="coerce") /
                pd.to_numeric(cohort[samp_col], errors="coerce").replace({0: np.nan})
            )
            weight_col = "__weight__"

    if weight_col:
        tmp = cohort[["stratum", weight_col]].copy().rename(columns={weight_col:"weight"})
        wmap = dict(zip(tmp["stratum"], tmp["weight"]))
    else:
        warn("cohort_table.csv has no 'weight' and cannot infer from N/n_sample; using flat weights = 1.0")
        wmap = {}

    sample["weight"] = sample["stratum"].map(lambda s: float(wmap.get(s, 1.0)))
    return sample

def index_local_configs_by_repo(root: Path) -> dict[str, list[Path]]:
    mapping: dict[str, list[Path]] = {}
    if not root.exists(): return mapping
    for p in root.rglob("*"):
        if not p.is_file(): continue
        if p.suffix.lower() not in TEXT_EXTS:
            if p.stat().st_size > 0 and p.suffix == "": pass
            else: continue
        full = full_name_from_saved_filename(p) or full_name_from_saved_filename(p.parent)
        if not full: continue
        mapping.setdefault(full, []).append(p)
    return mapping

# ======================= MAIN =======================
def main():
    start_t = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    banner("SETUP")
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    rot = TokenRotator(tokens)
    if tokens: info(f"Loaded {len(tokens)} GitHub token(s) for rotation from {TOKENS_ENV_PATH}")
    else:      warn("No tokens found; unauthenticated fetch will work only for public repos and low rate limits.")

    # time window
    start_arg = end_arg = None
    date_only = False
    if USE_CUTOFF:
        start_arg, end_arg, date_only = window_utc_from_local(CLONE_DATE_LOCAL, WINDOW_DAYS_BEFORE, WINDOW_DAYS_AFTER, LOCAL_TZ)
        if start_arg is None and end_arg is None:
            warn(f"Invalid cutoff '{CLONE_DATE_LOCAL}', proceeding without a cutoff.")
        else:
            if date_only:
                info(f"Applying DATE window: [{start_arg} .. {end_arg}] (no tz support available)")
            else:
                info(f"Applying UTC window: [{start_arg.isoformat()} .. {end_arg.isoformat()}] "
                     f"(local {LOCAL_TZ}, before={WINDOW_DAYS_BEFORE}, after={WINDOW_DAYS_AFTER})")

    banner("LOAD SAMPLE + WEIGHTS")
    sample_df = load_sample_and_weights(SAMPLED_CSV_DIR, SAMPLED_CSV_NAME, COHORT_FILE_NAME)
    sample_df = sample_df[~sample_df["full_name"].isna()].copy()
    repo_list = sorted(sample_df["full_name"].str.lower().unique().tolist())
    info(f"Repos in stratified sample: {len(repo_list)}")

    banner("INDEX LOCAL CONFIGS FOR YAML HINTS")
    repo_to_files = index_local_configs_by_repo(CONFIG_DIR)
    info(f"Repos with any local files detected: {len(repo_to_files)}")

    banner("FETCH & SCAN")
    rows, examples, audit = [], [], []
    total_runs_fetched = 0
    repos_with_hits = 0

    for idx, full in enumerate(repo_list, start=1):
        tic = time.time()
        provider = "github_actions"
        sub = sample_df[sample_df["full_name"].str.lower() == full]
        stratum = sub["stratum"].iloc[0] if not sub.empty else "Y0_B0_A0"
        weight  = float(sub["weight"].mean()) if not sub.empty else 1.0
        at_pred = int(sub["AT_pred"].max()) if ("AT_pred" in sub.columns and not sub["AT_pred"].isna().all()) else 0
        owner, repo = full.split("/", 1)

        # YAML hints (context only)
        yaml_text, wf_err = ("", None)
        if ENABLE_GITHUB_FETCH:
            yaml_text, wf_err = fetch_workflow_yamls(rot, owner, repo)
        local_files = repo_to_files.get(full, [])
        for fp in local_files:
            try:
                with open(fp, "r", encoding="utf-8", errors="ignore") as f:
                    yaml_text += "\n" + f.read()
            except Exception:
                pass
        yaml_hints = scan_yaml_hints(yaml_text) if yaml_text else []

        # STRATUM-based instrumentation context
        instru_ctx = has_instru_context_from_stratum(stratum, at_pred)

        # Eligible themes for counting
        eligible_themes = set(HIGH_THEMES)
        if INCLUDE_MEDIUM and instru_ctx:
            eligible_themes |= MEDIUM_THEMES
        if INCLUDE_LOW and instru_ctx:
            eligible_themes |= LOW_THEMES

        # Accumulators for de-noising (lines + runs)
        theme_line_hits: dict[str, int] = {}
        theme_run_ids:  dict[str, set]  = {}
        kept_examples = []  # (theme, run_id, line)

        # Iterate runs with pagination
        runs_err = None
        page, got = 1, 0
        while got < MAX_RUNS_PER_REPO:
            runs_resp = list_workflow_runs(rot, owner, repo, page=page, per_page=min(100, MAX_RUNS_PER_REPO))
            if runs_resp.status_code != 200:
                runs_err = runs_resp._diag; break
            wr = runs_resp.json().get("workflow_runs", [])
            if not wr: break

            for run in wr:
                if got >= MAX_RUNS_PER_REPO: break

                # date window filter
                created = parse_gh_ts(run.get("created_at"))
                if (start_arg is not None) or (end_arg is not None):
                    if date_only:
                        d = created.date() if created else None
                        if start_arg is not None and d is not None and d < start_arg: 
                            continue
                        if end_arg is not None and d is not None and d > end_arg: 
                            continue
                    else:
                        if start_arg is not None and created and created < start_arg: 
                            continue
                        if end_arg is not None and created and created > end_arg: 
                            continue

                # fetch logs for this run
                txt, diag = fetch_log_text_for_run(rot, run.get("logs_url"))
                if diag is not None:
                    runs_err = diag
                    continue
                got += 1
                total_runs_fetched += 1

                # scan
                fthemes, fhits = scan_text_for_themes(txt)

                # Per-theme accumulation (lines + run ids) only for eligible themes
                # Count line hits per theme (across all lines in this run)
                per_theme_line_count = {}
                for th, line in fhits:
                    if th in eligible_themes:
                        per_theme_line_count[th] = per_theme_line_count.get(th, 0) + 1

                for th, line_count in per_theme_line_count.items():
                    theme_line_hits[th] = theme_line_hits.get(th, 0) + line_count
                    theme_run_ids.setdefault(th, set()).add(run["id"])

                # Keep up to 3 example snippets across eligible themes
                if fhits and len(kept_examples) < 3:
                    for th, line in fhits:
                        if th in eligible_themes:
                            kept_examples.append((th, run["id"], line))
                            if len(kept_examples) >= 3: break

                # Early stop logic
                if EARLY_STOP_ON_HIT:
                    if EARLY_STOP_MODE == "any":
                        if any(_theme_counts_as_hit(th, theme_line_hits, theme_run_ids) for th in eligible_themes):
                            break
                    elif EARLY_STOP_MODE == "high-only":
                        if any(th in HIGH_THEMES and _theme_counts_as_hit(th, theme_line_hits, theme_run_ids, high=True)
                               for th in eligible_themes):
                            break

            # early stop outer check
            if EARLY_STOP_ON_HIT:
                if EARLY_STOP_MODE == "any":
                    if any(_theme_counts_as_hit(th, theme_line_hits, theme_run_ids) for th in eligible_themes):
                        break
                elif EARLY_STOP_MODE == "high-only":
                    if any(th in HIGH_THEMES and _theme_counts_as_hit(th, theme_line_hits, theme_run_ids, high=True)
                           for th in eligible_themes):
                        break

            page += 1

        # Final counted themes after de-noising
        seen_themes = set()
        for th in eligible_themes:
            if _theme_counts_as_hit(th, theme_line_hits, theme_run_ids):
                seen_themes.add(th)

        if seen_themes: repos_with_hits += 1

        # examples (only for themes that ultimately counted)
        for th, run_id, snip in kept_examples:
            if th in seen_themes:
                examples.append({
                    "full_name": full, "provider": provider, "stratum": stratum,
                    "theme": th, "run_id": run_id, "snippet": (snip or "").strip()[:500]
                })

        # row & audit
        rows.append({
            "full_name": full,
            "provider": provider,
            "stratum": stratum,
            "weight": weight,
            "themes": ";".join(sorted(seen_themes)) if seen_themes else "",
            "yaml_hints": ";".join(sorted(set(yaml_hints))) if yaml_hints else "",
            "log_runs_scanned": got,
            "yaml_bytes": len(yaml_text),
            "runs_err": json.dumps(runs_err) if runs_err else "",
            "wf_err": json.dumps(wf_err) if wf_err else "",
        })

        audit.append({
            "full_name": full,
            "log_runs_scanned": got,
            "yaml_bytes": len(yaml_text),
            "yaml_hints": ";".join(sorted(set(yaml_hints))) if yaml_hints else "",
            "themes_high_only": ";".join(sorted([t for t in seen_themes if t in HIGH_THEMES])) if seen_themes else "",
            "rate_remaining": (runs_err or {}).get("rate_remaining") if runs_err else None,
        })

        # progress line
        if idx % PROGRESS_EVERY == 0:
            dur = time.time() - tic
            err_flag = " ERR" if (rows[-1]["runs_err"] or rows[-1]["wf_err"]) else ""
            snip_ct = len([1 for th, _, _ in kept_examples if th in seen_themes]) if SHOW_HIT_SNIPPET_COUNT else 0
            note(f"[{idx}/{len(repo_list)}]{err_flag} {full} | runs={rows[-1]['log_runs_scanned']} | themes={len(seen_themes)}"
                 + (f" | examples={snip_ct}" if SHOW_HIT_SNIPPET_COUNT else "")
                 + f" | hints={len(yaml_hints)} | {dur:.1f}s")

    # Build outputs
    df = pd.DataFrame(rows)

    # Explode themes
    df_exp = df.copy()
    df_exp["themes"] = df_exp["themes"].fillna("")
    df_exp = df_exp[df_exp["themes"] != ""]
    if not df_exp.empty:
        df_exp = df_exp.assign(theme=df_exp["themes"].str.split(";")).explode("theme")
    else:
        df_exp = pd.DataFrame(columns=["full_name","provider","stratum","weight","theme"])

    # Weighted counts
    if not df_exp.empty:
        grp = df_exp.groupby("theme").agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        total_w = float(np.nansum(df["weight"].values))
        grp["share"] = np.where(total_w>0, grp["weighted_count"]/total_w, np.nan)
        grp = grp.sort_values(["weighted_count","repos"], ascending=[False, False])
    else:
        grp = pd.DataFrame(columns=["theme","weighted_count","repos","share"])

    # By provider
    if not df_exp.empty:
        byprov = df_exp.groupby(["provider","theme"]).agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        totals = byprov.groupby("provider")["weighted_count"].sum().rename("total_w")
        byprov = byprov.merge(totals, on="provider", how="left")
        byprov["share_in_provider"] = np.where(byprov["total_w"]>0, byprov["weighted_count"]/byprov["total_w"], np.nan)
        byprov = byprov.drop(columns=["total_w"]).sort_values(["provider","weighted_count"], ascending=[True, False])
    else:
        byprov = pd.DataFrame(columns=["provider","theme","weighted_count","repos","share_in_provider"])

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    grp.to_csv(OUT_DIR/"challenge_counts_weighted_high_only.csv", index=False, encoding="utf-8-sig")
    byprov.to_csv(OUT_DIR/"challenge_counts_by_provider_weighted_high_only.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(examples).to_csv(OUT_DIR/"challenge_examples_high_only.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(audit).to_csv(OUT_DIR/"challenge_run_audit_high_only.csv", index=False, encoding="utf-8-sig")

    banner("DONE")
    info(f"Saved -> {OUT_DIR/'challenge_counts_weighted_high_only.csv'}")
    info(f"Saved -> {OUT_DIR/'challenge_counts_by_provider_weighted_high_only.csv'}")
    info(f"Saved -> {OUT_DIR/'challenge_examples_high_only.csv'}")
    info(f"Saved -> {OUT_DIR/'challenge_run_audit_high_only.csv'}")
    elapsed = time.time() - start_t
    note(f"SUMMARY: repos={len(repo_list)} | total_runs_fetched={total_runs_fetched} | repos_with_any_hits={sum(df['themes'].astype(bool))} | elapsed={elapsed:.1f}s")

if __name__ == "__main__":
    main()


[2025-08-26 19:22:51] ===== SETUP =====
[2025-08-26 19:22:51] [INFO] Loaded 6 GitHub token(s) for rotation from C:\GitHub\Android-Mobile-Apps\all_tokens.env
[2025-08-26 19:22:51] [INFO] Applying UTC window: [2025-05-12T04:00:00+00:00 .. 2025-08-11T03:59:59+00:00] (local America/Toronto, before=90, after=0)
[2025-08-26 19:22:51] ===== LOAD SAMPLE + WEIGHTS =====
[2025-08-26 19:22:51] [INFO] Repos in stratified sample: 377
[2025-08-26 19:22:51] ===== INDEX LOCAL CONFIGS FOR YAML HINTS =====
[2025-08-26 19:22:54] [INFO] Repos with any local files detected: 4519
[2025-08-26 19:22:54] ===== FETCH & SCAN =====
[2025-08-26 19:23:01] [1/377] ERR 100mslive/100ms-android | runs=3 | themes=0 | examples=0 | hints=1 | 7.1s
[2025-08-26 19:23:02] [2/377] 10miaomiao/bili-down-out | runs=0 | themes=0 | examples=0 | hints=1 | 0.9s
[2025-08-26 19:23:02] [3/377] ERR 7heaven/shswitchview | runs=0 | themes=0 | examples=0 | hints=0 | 0.4s
[2025-08-26 19:23:05] [4/377] a-mabe/openhiit | runs=1 | themes=1 | ex